In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import time

In [2]:
# ----------------------
# Data Augmentation (important)
# ----------------------
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

In [3]:
# ----------------------
# Datasets
# ----------------------
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_data  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

In [5]:
# ----------------------
# Model (Transfer Learning)
# ----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 10)  # CIFAR-10 has 10 classes
model = model.to(device)

In [6]:
# ----------------------
# Loss & Optimizer
# ----------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

In [7]:
# ----------------------
# Training Loop
# ----------------------
epochs = 30
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    start = time.time()

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    # Quick validation
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}]  Loss: {running_loss/len(train_loader):.4f}  Acc: {acc:.2f}%  Time: {time.time()-start:.1f}s")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "cifar10_resnet18.pth")
        print(f"  → New best model saved! ({best_acc:.2f}%)")

Epoch [1/30]  Loss: 1.0363  Acc: 69.29%  Time: 25.7s
  → New best model saved! (69.29%)
Epoch [2/30]  Loss: 0.7401  Acc: 77.17%  Time: 24.9s
  → New best model saved! (77.17%)
Epoch [3/30]  Loss: 0.6492  Acc: 78.71%  Time: 24.8s
  → New best model saved! (78.71%)
Epoch [4/30]  Loss: 0.5892  Acc: 77.52%  Time: 24.8s
Epoch [5/30]  Loss: 0.5419  Acc: 80.55%  Time: 24.5s
  → New best model saved! (80.55%)
Epoch [6/30]  Loss: 0.5096  Acc: 82.27%  Time: 24.8s
  → New best model saved! (82.27%)
Epoch [7/30]  Loss: 0.4887  Acc: 82.29%  Time: 25.1s
  → New best model saved! (82.29%)
Epoch [8/30]  Loss: 0.4569  Acc: 82.51%  Time: 25.7s
  → New best model saved! (82.51%)
Epoch [9/30]  Loss: 0.4230  Acc: 82.89%  Time: 25.5s
  → New best model saved! (82.89%)
Epoch [10/30]  Loss: 0.3959  Acc: 83.92%  Time: 26.6s
  → New best model saved! (83.92%)
Epoch [11/30]  Loss: 0.3781  Acc: 84.73%  Time: 26.6s
  → New best model saved! (84.73%)
Epoch [12/30]  Loss: 0.3615  Acc: 84.15%  Time: 26.2s
Epoch [13/3

In [8]:
print(f"\nTraining finished. Best accuracy: {best_acc:.2f}%")


Training finished. Best accuracy: 87.93%
